# 01 - Chess Environment and Encoding

Define the observable chess task, inspect the encodings, and run ordinary correctness tests. No trained model or measured playing strength is claimed by this notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT)
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Sources and Environment Contract

The reference snapshot is included. The referee retains move history; the agent sees FEN and its remaining clock. The reserved repetition plane is zero in training and runtime.

In [ ]:
source = read_json(PROJECT_ROOT / "docs/source_manifest.json")
print("Source date:", source["inspected_at"])
print("Official runtime versions:", source["runtime_versions"])
print((PROJECT_ROOT / "docs/ENVIRONMENT_CONTRACT.md").read_text())

## A Standard Chess Position

Every piece constraint comes from python-chess. These are legal UCI actions.

In [ ]:
import chess
from IPython.display import display
from chess_rl.environment import ChessEnvironment

environment = ChessEnvironment()
fen, remaining_ms = environment.observe()
display(environment.board)
print("FEN:", fen)
print("Remaining milliseconds:", remaining_ms)
print("Legal UCI moves:", [move.uci() for move in environment.board.legal_moves])

## Board and Action Representation

Absolute White-oriented coordinates are used for both players. The policy index is plane * 64 + from_square.

In [ ]:
from chess_rl.board_encoding import encode_board
from chess_rl.action_encoding import encode_move, decode_move, legal_mask

board = environment.board
encoded = encode_board(board)
move = chess.Move.from_uci("e2e4")
index = encode_move(board, move)
print("Input shape:", tuple(encoded.shape))
print("e2e4 index:", index, "| round-trip:", decode_move(board, index).uci())
print("Legal-mask entries:", int(legal_mask(board).sum()))
print("History plane sum:", float(encoded[20].sum()))

## Correctness Tests

These cover rules, encodings, legal losses, timeout restoration, and checkpoint recovery. They are not submission compliance or long training runs.

In [ ]:
subprocess.check_call([sys.executable, "-m", "pytest", "-q", "tests"], cwd=PROJECT_ROOT)

## Versioned Opening Suites

Reserve independent legal playouts before the corpus. These are valid research fixtures, not certified balanced positions or the hidden platform set. The public eight remain in the reference harness.

In [ ]:
from chess_rl.dataset import prepare_openings
opening_manifest = prepare_openings(PROJECT_ROOT, seed=cfg["seed"])
print(opening_manifest)

## Next Notebook

Open notebook 02. An A100 accelerates gradient updates; Stockfish labels positions on CPU. Every notebook reloads persisted inputs from Drive.